# **BAN 612 PROJECT - Data Collection Notebook**

# VERSION 1 - Save to Google Spreadsheet

In [ ]:
import streamlit as st
import pandas as pd
from jobspy import scrape_jobs
from streamlit_gsheets import GSheetsConnection
from datetime import datetime

# --- CONFIGURATION & GOOGLE SHEETS SETUP ---
st.set_page_config(page_title="BAN 612 Job Collector", layout="wide")
st.title("🚀 BAN 612: Strategic Job Data Collector")

# Connection to a central Google Sheet (Database)
# Note: Requires setting up .streamlit/secrets.toml with your Sheet URL
conn = st.connection("gsheets", type=GSheetsConnection)

# --- USER INPUT SECTION ---
with st.sidebar:
    st.header("👤 User Information")
    user_name = st.text_input("Enter your name", placeholder="e.g., Thu Ha")

    st.header("🔍 Search Criteria")
    industries = st.multiselect("Industries (Max 3)",
                               ["High Tech", "Finance", "Consulting", "Healthcare", "Retail", "Real Estate"],
                               max_selections=3)

    companies = st.text_input("Specific Companies (Optional, comma-separated)", placeholder="e.g., McKinsey, Google")
    roles = st.text_input("Target Roles (e.g., Financial Consultant)", placeholder="Required")
    skills = st.text_input("Target Skills (Optional)", placeholder="e.g., Python, SQL")

    entry_limit = st.number_input("Entries to collect per run", min_value=10, max_value=200, value=50)

# --- THE SEARCH ENGINE ---
if st.button("▶️ Run Automated Collection"):
    if not user_name or not roles:
        st.error("Please provide your name and the target roles.")
    else:
        with st.status("Searching giant job-boards...", expanded=True) as status:
            try:
                # Format company list if provided
                company_list = [c.strip() for c in companies.split(",")] if companies else None

                # EXECUTE SCRAPE (JobSpy handles LinkedIn, Indeed, Glassdoor, etc.)
                jobs = scrape_jobs(
                    site_name=["linkedin", "indeed", "glassdoor", "zip_recruiter"],
                    search_term=roles,
                    location="USA",
                    results_wanted=entry_limit,
                    hours_old=72, # Fresh listings only
                    company_prohibited_words=None,
                    # Optional: filter by company list if provided
                )

                if not jobs.empty:
                    # Enrich data with metadata
                    jobs['Collector_Name'] = user_name
                    jobs['Search_Industry'] = ", ".join(industries)
                    jobs['Collection_Timestamp'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                    # 1. Fetch existing data from Google Sheets to prevent duplicates
                    existing_data = conn.read()

                    # 2. Append new data
                    updated_df = pd.concat([existing_data, jobs], ignore_index=True)

                    # 3. Save back to Google Sheets (Centralized Backup)
                    conn.update(data=updated_df)

                    status.update(label="Collection Complete! Data saved to central database.", state="complete")
                    st.success(f"Added {len(jobs)} new entries.")
                else:
                    st.warning("No jobs found for these criteria.")
            except Exception as e:
                st.error(f"Error: {e}")

# --- DATA VIEW & EXPORT ---
st.divider()
st.subheader("📊 Centralized Project Dataset")

# Read the latest state of the master dataset
try:
    master_df = conn.read()

    # Display the data (Read-Only)
    st.dataframe(master_df, use_container_width=True)

    col1, col2 = st.columns(2)
    with col1:
        # Export option for current user's work
        my_work = master_df[master_df['Collector_Name'] == user_name]
        st.download_button("📥 Download My Collected Data (CSV)",
                           data=my_work.to_csv(index=False),
                           file_name=f"my_work_{user_name}.csv")

    with col2:
        # Export option for the whole team (Full Backup)
        st.download_button("📥 Download FULL Dataset (CSV Backup)",
                           data=master_df.to_csv(index=False),
                           file_name="BAN612_Full_Dataset.csv")

except:
    st.info("The central database is currently empty. Run your first search to begin!")